In [37]:
import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import shap
import warnings
from datetime import datetime
import io
warnings.filterwarnings('ignore')

llm_pipeline = None

In [23]:

#  CSS

custom_css = """
/* --- GLOBAL LIGHT BACKGROUND --- */
body, .gradio-container {
    background: #f4f7fc !important;
    font-family: 'Inter', 'Segoe UI', 'Helvetica Neue', sans-serif !important;
}
.gr-box, .gradio-interface, .tabs {
    background: #ffffff !important;
    border-radius: 16px !important;
    box-shadow: 0 2px 12px rgba(0, 0, 0, 0.03) !important;
    border: 1px solid #e9eef4 !important;
    padding: 20px !important;
}

/* --- BOLD HEADERS & TITLES --- */
h1, h2, h3, .gr-label {
    font-weight: 700 !important;
    color: #0b1a33 !important;
}
.title-gradient {
    font-size: 2.5rem !important;
    font-weight: 800 !important;
    color: #0b1a33 !important;
    margin-bottom: 0 !important;
}
.subtitle {
    font-weight: 600 !important;
    color: #4a6a8a !important;
    border-bottom: 4px solid #0066cc;
    padding-bottom: 6px;
    display: inline-block;
}

/* --- METRIC CARDS (TITLES ARE FULLY VISIBLE) --- */
.metric-card {
    background: #ffffff !important;
    border-radius: 12px !important;
    padding: 14px 18px !important;
    border: 1px solid #dfe6ef !important;
    box-shadow: 0 2px 8px rgba(0,0,0,0.02) !important;
    height: 100% !important;
}
.metric-card:hover {
    border-color: #0066cc !important;
    box-shadow: 0 4px 16px rgba(0, 102, 204, 0.08) !important;
}
.metric-label {
    font-weight: 700 !important;
    color: #5a6d86 !important;
    font-size: 0.75rem !important;
    text-transform: uppercase !important;
    letter-spacing: 1px !important;
    display: block !important;
    margin-bottom: 2px !important;
}
.metric-value {
    font-weight: 800 !important;
    font-size: 2.2rem !important;
    color: #0b1a33 !important;
    line-height: 1.2 !important;
}

/* --- PRIMARY ACTION BUTTON --- */
.primary-btn {
    background: #0066cc !important;
    border: none !important;
    color: white !important;
    font-weight: 700 !important;
    padding: 12px 32px !important;
    border-radius: 40px !important;
    box-shadow: 0 4px 14px rgba(0, 102, 204, 0.25) !important;
    transition: all 0.2s !important;
}
.primary-btn:hover {
    background: #004b99 !important;
    transform: translateY(-2px);
    box-shadow: 0 8px 24px rgba(0, 102, 204, 0.35) !important;
}

/* --- TABS STYLING --- */
.tab-nav button {
    font-weight: 700 !important;
    color: #4a5b6e !important;
    padding: 10px 22px !important;
}
.tab-nav button.selected {
    background: #0066cc !important;
    color: white !important;
    border-radius: 8px !important;
}

/* --- FOOTER --- */
.footer-text {
    font-weight: 500 !important;
    color: #8895aa !important;
    text-align: center !important;
    border-top: 2px solid #eef2f7 !important;
    padding-top: 20px !important;
    margin-top: 20px !important;
}
"""


In [36]:

#  DATA PREPROCESSING ENGINE

def preprocess_logs(df):
    """
    Automatically detects features, handles timestamps, 
    and enriches data with behavioral features for ML.
    """
    df_clean = df.copy()
    
    # ----  Detect Timestamp Column ----
    time_col = None
    for col in df_clean.columns:
        if 'time' in col.lower() or 'date' in col.lower() or 'timestamp' in col.lower():
            try:
                df_clean[col] = pd.to_datetime(df_clean[col])
                time_col = col
            except:
                pass
    
    # ----  Enrich with Behavioral Features ----
    # Frequency analysis of source IPs (repeated IPs are suspicious)
    if 'src_ip' in df_clean.columns:
        ip_counts = df_clean['src_ip'].value_counts().to_dict()
        df_clean['src_ip_freq'] = df_clean['src_ip'].map(ip_counts)
    
    # Flag dangerous ports (commonly used in attacks)
    if 'dst_port' in df_clean.columns:
        dangerous_ports = [22, 23, 3389, 445, 4444, 1433, 3306]
        df_clean['is_dangerous_port'] = df_clean['dst_port'].isin(dangerous_ports).astype(int)
    
    # ----  Select Numerical Features for ML ----
    exclude = ['timestamp', 'date', 'time', 'id', 'label', 'attack', 
               'src_ip', 'dst_ip', 'protocol', 'url', 'domain']
    numeric_cols = []
    for col in df_clean.columns:
        if col.lower() not in [x.lower() for x in exclude]:
            try:
                if pd.api.types.is_numeric_dtype(df_clean[col]):
                    numeric_cols.append(col)
            except:
                pass
    
    # ----  Emergency Fallback (if no numeric columns found) ----
    if len(numeric_cols) < 2:
        if 'packet_size' not in df_clean.columns:
            df_clean['synthetic_bytes'] = np.random.randint(100, 1500, len(df_clean))
            numeric_cols.append('synthetic_bytes')
        if 'src_port' in df_clean.columns:
            numeric_cols.append('src_port')
        if 'dst_port' in df_clean.columns:
            numeric_cols.append('dst_port')
        if 'packet_count' in df_clean.columns:
            numeric_cols.append('packet_count')
    
    print(f"✅ Preprocessing complete. Features: {numeric_cols[:5]}... (total {len(numeric_cols)})")
    return df_clean, numeric_cols, time_col


In [35]:

#  TRAINING & ANOMALY DETECTION ENGINE

def train_and_detect(df, numeric_cols):
    """
    Trains Isolation Forest on the data and returns predictions.
    """
    # ----  Prepare Feature Matrix ----
    X = df[numeric_cols].fillna(0)
    
    # ----  Standardize Features ----
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # ----  Train Isolation Forest ----
    model = IsolationForest(
        contamination=0.15,      # Assume 15% of data is anomalous
        random_state=42,
        n_estimators=100,
        max_samples='auto'
    )
    model.fit(X_scaled)
    
    # ----  Predict ----
    preds = model.predict(X_scaled)   # -1 = anomaly, 1 = normal
    scores = model.decision_function(X_scaled)  # Lower = more anomalous
    
    # ----  Create Results DataFrame ----
    df_results = df.copy()
    df_results['anomaly_score'] = scores
    df_results['is_anomaly'] = preds == -1
    
    print(f"✅ Model trained. Anomalies detected: {df_results['is_anomaly'].sum()}")
    return model, scaler, df_results, X_scaled


In [26]:

#  SHAP EXPLAINABILITY ENGINE

def get_shap_explanations(model, X_scaled, numeric_cols, df_results):
    """
    Calculates SHAP values for the top 3 anomalous events
    to explain which features contributed most to the anomaly.
    """
    explanations = []
    
    # Only run if there is at least one anomaly
    if df_results['is_anomaly'].sum() > 0:
        try:
            # Use a subset for speed (max 500 rows)
            subset_X = X_scaled[:min(500, len(X_scaled))]
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(subset_X)
            
            # Get indices of the top 3 anomalies
            anomaly_indices = df_results[df_results['is_anomaly']].index[:3]
            
            for idx in anomaly_indices:
                if idx < len(shap_values):
                    vals = shap_values[idx]
                    # Sort features by absolute SHAP value
                    top_features = sorted(
                        [(numeric_cols[i], abs(vals[i])) for i in range(len(vals))],
                        key=lambda x: x[1], 
                        reverse=True
                    )[:3]
                    explanations.append({
                        "row": int(idx),
                        "top_features": [f"{feat}: {val:.4f}" for feat, val in top_features]
                    })
        except Exception as e:
            explanations = [{"error": f"SHAP explanation limited: {str(e)}"}]
    else:
        explanations = [{"info": "No anomalies detected to explain."}]
    
    return explanations


In [34]:

#  AI INCIDENT REPORT GENERATOR (LLM + Fallback)

def generate_report(df_results, numeric_cols, time_col):
    """
    Generates a professional incident report with ranked suspects and response actions.
    """
    total = len(df_results)
    anomalies = df_results['is_anomaly'].sum()
    risk = round((anomalies / total) * 100, 1) if total > 0 else 0
    
    # ----  Rank Threat Suspects (Top 5) ----
    ranked_suspects = []
    if 'src_ip' in df_results.columns and anomalies > 0:
        suspect_counts = df_results[df_results['is_anomaly']]['src_ip'].value_counts().head(5)
        for i, (ip, count) in enumerate(suspect_counts.items(), 1):
            ranked_suspects.append(f"{i}. **{ip}** ({count} anomalous events)")
    
    suspects_markdown = "**🎯 Top 5 Threat Suspects (Ranked)**\n"
    if ranked_suspects:
        suspects_markdown += "\n".join(ranked_suspects)
    else:
        suspects_markdown += "✅ No threats detected. Network appears clean."
    
    # ----  Generate Response Actions ----
    actions_markdown = """
    **🚨 Recommended Incident Response Actions**
    | Priority | Action |
    | :--- | :--- |
    | **Immediate** | **Containment**: Isolate top suspect IPs via firewall ACLs to prevent lateral movement. |
    | **High** | **Investigation**: Perform deep packet inspection (DPI) on outbound traffic from compromised hosts. |
    | **Medium** | **Recovery**: Reset credentials and force MFA re-enrollment for affected user accounts. |
    | **Low** | **Hardening**: Review and update IDS/IPS rules to block similar patterns in the future. |
    """
    
    # ---- Try to load LLM (lazy loading) ----
    global llm_pipeline
    if llm_pipeline is None:
        try:
            from transformers import pipeline
            llm_pipeline = pipeline(
                "text-generation", 
                model="microsoft/phi-2", 
                device_map="auto", 
                max_new_tokens=150, 
                trust_remote_code=True
            )
        except:
            llm_pipeline = False
    
    # ----  Generate with LLM if available ----
    if llm_pipeline and llm_pipeline != False:
        top_ip = ranked_suspects[0].split('**')[1] if ranked_suspects else "N/A"
        prompt = f"""
        Cybersecurity incident analysis:
        Total logs: {total}
        Suspicious events: {anomalies} ({risk}% of traffic)
        Top suspect: {top_ip}
        Provide a 2-sentence executive summary focusing on the severity and potential impact.
        """
        try:
            resp = llm_pipeline(prompt, max_new_tokens=100, do_sample=True, temperature=0.6)
            llm_summary = resp[0]['generated_text'].replace(prompt, "").strip()
        except:
            llm_summary = ""
    else:
        llm_summary = ""
    
    # ----  Final Professional Report ----
    severity = "🔴 CRITICAL" if risk > 20 else "🟠 HIGH" if risk > 10 else "🟡 MEDIUM" if risk > 5 else "🟢 LOW"
    
    final_report = f"""
    ## 📋 Incident Report Summary

    **Severity**: {severity}  
    **Total Events Analyzed**: {total}  
    **Suspicious Alerts**: {anomalies} (🔴 {risk}% of traffic)  

    ---
    {llm_summary if llm_summary else f"**Executive Summary**: {anomalies} suspicious events detected across {len(ranked_suspects)} unique source IPs. Immediate containment of the top suspect is advised to prevent potential data exfiltration."}

    ---
    {suspects_markdown}

    ---
    {actions_markdown}
    """
    return final_report


In [32]:

#  PLOTTING ENGINE 

def create_unique_plots(df_results, numeric_cols, time_col):
    """
    Generates 4 unique, judge-impressing visualizations with CLEAR BOLD FONTS.
    """
    plots = {}
    
    # ----  Threat Timeline ----
    if time_col and time_col in df_results.columns:
        fig1 = px.scatter(
            df_results, 
            x=time_col, 
            y='anomaly_score', 
            color='is_anomaly',
            title="⏳ Threat Timeline",
            labels={'anomaly_score': 'Suspicion Score', time_col: 'Time'},
            color_discrete_map={True: '#d32f2f', False: '#1976d2'},
            template="plotly_white"
        )
        fig1.update_traces(marker=dict(size=12, line=dict(width=1.5, color='white')))
    else:
        fig1 = px.histogram(
            df_results, 
            x='anomaly_score', 
            color='is_anomaly',
            title="📊 Anomaly Score Distribution",
            template="plotly_white",
            color_discrete_map={True: '#d32f2f', False: '#1976d2'}
        )
    # --- FONT UPGRADE ---
    fig1.update_layout(
        font=dict(size=16, family="Inter, Arial, sans-serif", color="#0b1a33"),
        title_font=dict(size=20, family="Inter, Arial, sans-serif", color="#0b1a33"),
        legend_font=dict(size=14),
        xaxis_title_font=dict(size=16, family="Inter, Arial, sans-serif"),
        yaxis_title_font=dict(size=16, family="Inter, Arial, sans-serif"),
        xaxis_tickfont=dict(size=13),
        yaxis_tickfont=dict(size=13)
    )
    plots['timeline'] = fig1

    # ----  Protocol Breakdown (Pie Chart) ----
    if 'protocol' in df_results.columns:
        counts = df_results.groupby(['protocol', 'is_anomaly']).size().reset_index(name='count')
        fig2 = px.pie(
            counts, 
            values='count', 
            names='protocol', 
            title="🌐 Protocol Breakdown",
            color='is_anomaly',
            color_discrete_map={True: '#d32f2f', False: '#1976d2'},
            template="plotly_white"
        )
        fig2.update_traces(textposition='inside', textinfo='percent+label', textfont_size=16)
    elif len(numeric_cols) > 0:
        fig2 = px.box(
            df_results, 
            y=numeric_cols[0], 
            color='is_anomaly',
            title=f"🔬 {numeric_cols[0]} Distribution",
            template="plotly_white",
            color_discrete_map={True: '#d32f2f', False: '#1976d2'}
        )
    else:
        fig2 = px.scatter(title="Protocol data unavailable")
    # --- FONT UPGRADE ---
    fig2.update_layout(
        font=dict(size=16, family="Inter, Arial, sans-serif", color="#0b1a33"),
        title_font=dict(size=20, family="Inter, Arial, sans-serif", color="#0b1a33"),
        legend_font=dict(size=14)
    )
    plots['protocol'] = fig2

    # ----  Network Topology (Source vs Destination Frequency) ----
    if 'src_ip' in df_results.columns and 'dst_ip' in df_results.columns:
        src_counts = df_results['src_ip'].value_counts().head(10).reset_index()
        src_counts.columns = ['IP', 'Src_Count']
        dst_counts = df_results['dst_ip'].value_counts().head(10).reset_index()
        dst_counts.columns = ['IP', 'Dst_Count']
        ip_df = pd.merge(src_counts, dst_counts, on='IP', how='outer').fillna(0)
        
        suspicious = df_results[df_results['is_anomaly']]['src_ip'].unique()
        ip_df['is_suspect'] = ip_df['IP'].isin(suspicious)
        
        fig3 = px.scatter(
            ip_df, 
            x='Src_Count', 
            y='Dst_Count', 
            text='IP', 
            color='is_suspect',
            title="🌍 Network Topology: Source vs Destination Frequency",
            labels={'Src_Count': 'Outbound Connections', 'Dst_Count': 'Inbound Connections'},
            template="plotly_white",
            color_discrete_map={True: '#d32f2f', False: '#1976d2'}
        )
        fig3.update_traces(textposition='top center', marker=dict(size=20), textfont=dict(size=15))
    else:
        fig3 = px.scatter(title="IP columns missing from logs")
    # --- FONT UPGRADE ---
    fig3.update_layout(
        font=dict(size=16, family="Inter, Arial, sans-serif", color="#0b1a33"),
        title_font=dict(size=20, family="Inter, Arial, sans-serif", color="#0b1a33"),
        legend_font=dict(size=14),
        xaxis_title_font=dict(size=16),
        yaxis_title_font=dict(size=16)
    )
    plots['network'] = fig3

    # ----  Temporal Heatmap OR Correlation Matrix ----
    if time_col and time_col in df_results.columns:
        try:
            df_results['hour'] = df_results[time_col].dt.hour
            df_results['minute'] = df_results[time_col].dt.minute
            heat_data = df_results.pivot_table(
                index='hour', 
                columns='minute', 
                values='anomaly_score', 
                aggfunc='mean'
            ).fillna(0)
            fig4 = px.imshow(
                heat_data, 
                title="🕒 Attack Pattern Heatmap (Hour vs Minute)",
                labels=dict(x="Minute of Hour", y="Hour of Day", color="Avg Suspicion"),
                color_continuous_scale="Reds",
                template="plotly_white"
            )
            fig4.update_layout(height=400)
        except:
            fig4 = px.scatter(title="Heatmap requires datetime column")
    else:
        if len(numeric_cols) > 1:
            corr = df_results[numeric_cols].corr()
            fig4 = px.imshow(
                corr, 
                text_auto=True, 
                title="📈 Feature Correlation Matrix",
                color_continuous_scale="Blues",
                template="plotly_white"
            )
        else:
            fig4 = px.scatter(title="Not enough features for correlation")
    # --- FONT UPGRADE ---
    fig4.update_layout(
        font=dict(size=14, family="Inter, Arial, sans-serif", color="#0b1a33"),
        title_font=dict(size=20, family="Inter, Arial, sans-serif", color="#0b1a33"),
        xaxis_tickfont=dict(size=12),
        yaxis_tickfont=dict(size=12)
    )
    plots['heatmap'] = fig4
    
    return plots
print("✅ Block 7 replaced: Clear bold fonts added to all plots.")

✅ Block 7 replaced: Clear bold fonts added to all plots.


In [31]:

#  GRADIO UI, MAIN PROCESSING FUNCTION & LAUNCH


# ---------- Main Processing Function ----------
import os


def process_file(file, state):
    """
    Orchestrates the entire pipeline with DYNAMIC METRIC CARD UPDATES.
    """
    # Initial default values for cards
    default_metrics = {
        'total': '<div class="metric-card"><span class="metric-label">📋 TOTAL EVENTS</span><div class="metric-value" style="color:#1976d2;">0</div></div>',
        'anom': '<div class="metric-card"><span class="metric-label">⚠️ ANOMALIES</span><div class="metric-value" style="color:#d32f2f;">0</div></div>',
        'risk': '<div class="metric-card"><span class="metric-label">🔥 RISK SCORE</span><div class="metric-value" style="color:#f57c00;">0%</div></div>',
        'sus': '<div class="metric-card"><span class="metric-label">🎯 TOP SUSPECT</span><div class="metric-value" style="font-size:1.5rem; color:#004b99;">N/A</div></div>'
    }
    
    if file is None:
        return (state, "⏳ Awaiting upload...", "", "", 
                None, None, None, None, None, 
                default_metrics['total'], default_metrics['anom'], 
                default_metrics['risk'], default_metrics['sus'])
    
    try:
        # ---- Step 1: Read CSV ----
        df = pd.read_csv(file.name)
        if df.empty:
            return (state, "❌ File is empty.", "", "", 
                    None, None, None, None, None,
                    default_metrics['total'], default_metrics['anom'], 
                    default_metrics['risk'], default_metrics['sus'])
        
        # ---- Step 2: Preprocess ----
        df_proc, num_cols, time_col = preprocess_logs(df)
        
        # ---- Step 3: Train & Detect ----
        model, scaler, df_res, X_scaled = train_and_detect(df_proc, num_cols)
        
        # ---- Step 4: SHAP Explainability ----
        shap_data = get_shap_explanations(model, X_scaled, num_cols, df_res)
        
        # ---- Step 5: Generate AI Report (now includes ranked suspects) ----
        report = generate_report(df_res, num_cols, time_col)
        
        # ---- Step 6: Generate 4 Unique Plots ----
        plots = create_unique_plots(df_res, num_cols, time_col)
        
        # ---- Step 7: Extract Metrics ----
        total = len(df_res)
        anomalies = df_res['is_anomaly'].sum()
        risk = round((anomalies / total) * 100, 1) if total > 0 else 0
        
        # Determine top suspect for the card
        top_ip = "N/A"
        if 'src_ip' in df_res.columns and anomalies > 0:
            top_ip = df_res[df_res['is_anomaly']]['src_ip'].value_counts().index[0]
        
        # Store state
        state = {"model": model, "scaler": scaler, "features": num_cols}
        
        # ---- Step 8: Format Summary (Includes Ranked List) ----
        # Extract ranked suspects from the report or compute directly
        suspect_list = []
        if 'src_ip' in df_res.columns and anomalies > 0:
            suspect_counts = df_res[df_res['is_anomaly']]['src_ip'].value_counts().head(5)
            for i, (ip, count) in enumerate(suspect_counts.items(), 1):
                suspect_list.append(f"{i}. `{ip}` ({count} alerts)")
        
        suspects_md = "\n".join(suspect_list) if suspect_list else "✅ No threats detected."
        
        summary_md = f"""
        **📊 Executive Summary**  
        - **Total Events Analyzed**: `{total}`  
        - **Suspicious Alerts**: `{anomalies}` (🔴 {risk}% of traffic)  
        - **Primary Threat Actor**: `{top_ip}`  
        - **Analysis Status**: ✅ Complete  

        **🎯 Ranked Threat Suspects**  
        {suspects_md}
        """
        
        # ---- Step 9: Build Dynamic Metric Card HTML ----
        total_html = f'<div class="metric-card"><span class="metric-label">📋 TOTAL EVENTS</span><div class="metric-value" style="color:#1976d2;">{total}</div></div>'
        anom_html = f'<div class="metric-card"><span class="metric-label">⚠️ ANOMALIES</span><div class="metric-value" style="color:#d32f2f;">{anomalies}</div></div>'
        risk_html = f'<div class="metric-card"><span class="metric-label">🔥 RISK SCORE</span><div class="metric-value" style="color:#f57c00;">{risk}%</div></div>'
        sus_html = f'<div class="metric-card"><span class="metric-label">🎯 TOP SUSPECT</span><div class="metric-value" style="font-size:1.5rem; color:#004b99;">{top_ip}</div></div>'
        
        # ---- Step 10: Return 13 Values (matches UI outputs) ----
        return (state, summary_md, report, 
                plots['timeline'], plots['protocol'], plots['network'], plots['heatmap'],
                df_res.head(10), shap_data,
                total_html, anom_html, risk_html, sus_html)
        
    except Exception as e:
        return (state, f"❌ Error: {str(e)}", "", "", 
                None, None, None, None, None,
                default_metrics['total'], default_metrics['anom'], 
                default_metrics['risk'], default_metrics['sus'])

# ----------  Build the Gradio UI ----------
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"), css=custom_css) as demo:
    
    # ---- Header ----
    gr.HTML("""
    <div style="display: flex; justify-content: space-between; align-items: center;">
        <div>
            <h1 class="title-gradient">🔍 CogniTrace · DFIR</h1>
            <p class="subtitle">AI-Powered Digital Forensics & Incident Response</p>
        </div>
        <div style="background: #e6f0ff; padding: 6px 24px; border-radius: 40px;">
            <span style="font-weight: 700; color: #0066cc;">🔴 LIVE THREAT HUNT</span>
        </div>
    </div>
    """)
    
    # ---- Upload Row ----
    with gr.Row():
        with gr.Column(scale=3):
            file_input = gr.File(label="📁 Upload Network Logs (CSV)", file_types=[".csv", ".log"])
        with gr.Column(scale=1, min_width=160):
            analyze_btn = gr.Button("🚀 Execute Forensic Scan", variant="primary", elem_classes="primary-btn")
    
    # ---- Metric Cards (NOW DYNAMIC via gr.HTML) ----
    with gr.Row():
        with gr.Column(scale=1, min_width=120):
            total_card = gr.HTML('<div class="metric-card"><span class="metric-label">📋 TOTAL EVENTS</span><div class="metric-value" style="color:#1976d2;">0</div></div>')
        with gr.Column(scale=1, min_width=120):
            anom_card = gr.HTML('<div class="metric-card"><span class="metric-label">⚠️ ANOMALIES</span><div class="metric-value" style="color:#d32f2f;">0</div></div>')
        with gr.Column(scale=1, min_width=120):
            risk_card = gr.HTML('<div class="metric-card"><span class="metric-label">🔥 RISK SCORE</span><div class="metric-value" style="color:#f57c00;">0%</div></div>')
        with gr.Column(scale=1, min_width=120):
            sus_card = gr.HTML('<div class="metric-card"><span class="metric-label">🎯 TOP SUSPECT</span><div class="metric-value" style="font-size:1.5rem; color:#004b99;">N/A</div></div>')
    
    # ---- Tabs ----
    with gr.Tabs():
        # Tab 1: Executive Dashboard
        with gr.TabItem("📈 Executive Dashboard"):
            with gr.Row():
                with gr.Column(scale=1):
                    summary_box = gr.Markdown("**⏳ Awaiting upload...**")
                    gr.Markdown("---")
                    report_box = gr.Markdown("### 🧠 AI Incident Report\nClick 'Execute' to generate.")
                with gr.Column(scale=2):
                    plot_timeline = gr.Plot(label="⏳ Threat Timeline")
            with gr.Row():
                plot_protocol = gr.Plot(label="🌐 Protocol Breakdown")
        
        # Tab 2: Network Forensics (UNIQUE)
        with gr.TabItem("🌍 Network Forensics"):
            with gr.Row():
                plot_network = gr.Plot(label="🌍 Network Topology (Source vs Destination)")
            with gr.Row():
                plot_heatmap = gr.Plot(label="🕒 Temporal Attack Heatmap / Correlation")
        
        # Tab 3: Explainability (SHAP)
        with gr.TabItem("🔬 Explainability (SHAP)"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("""
                    **🔍 SHAP Analysis**  
                    This shows the top 3 numerical features driving the anomaly score for the most suspicious events.
                    """)
                with gr.Column(scale=2):
                    shap_output = gr.JSON(label="Top Feature Contributions")
        
        # Tab 4: Raw Data
        with gr.TabItem("📄 Raw Data"):
            raw_table = gr.Dataframe(label="Top 10 Log Entries (Anomalies Highlighted)", interactive=False)
    
    # ---- Footer ----
    gr.HTML("""
    <div class="footer-text">
        🏆 Cyber Defence Innovation Hackathon 2026 · Isolation Forest · SHAP · GenAI 
        <span style="float: right; font-weight: 700;">Enterprise v3.0</span>
    </div>
    """)
    
    # ----------  Wiring (Connect Button to Outputs) ----------
    # Now outputs 9-12 are the gr.HTML components for the cards!
    outputs = [
        gr.State(),      # 0: Model state
        summary_box,     # 1: Summary markdown
        report_box,      # 2: Report markdown
        plot_timeline,   # 3: Timeline plot
        plot_protocol,   # 4: Protocol plot
        plot_network,    # 5: Network plot
        plot_heatmap,    # 6: Heatmap/Correlation plot
        raw_table,       # 7: Data table
        shap_output,     # 8: SHAP JSON
        total_card,      # 9: TOTAL EVENTS (HTML)
        anom_card,       # 10: ANOMALIES (HTML)
        risk_card,       # 11: RISK SCORE (HTML)
        sus_card         # 12: TOP SUSPECT (HTML)
    ]
    
    analyze_btn.click(
        fn=process_file,
        inputs=[file_input, gr.State(value=None)],
        outputs=outputs
    )

    
    print("\n🚀 Launching CogniTrace Enterprise v3.0...")
    print("📎 Upload sample_network_logs.csv or your own log file.")
    demo.launch(share=True)



🚀 Launching CogniTrace Enterprise v3.0...
📎 Upload sample_network_logs.csv or your own log file.
* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://ebb9d5da072664ce1f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
